In [151]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
accuracy_score, f1_score, log_loss,
confusion_matrix, classification_report
)
from sklearn.base import clone



In [152]:
# Importing the cleaned data
Sleep_clean = pd.read_csv('../Kaggle_cleaning_eda/sleep_clean.csv')

In [153]:
Sleep_clean.columns

Index(['age', 'bedtime_screen_time_minutes', 'total_daily_screen_time_hours',
       'doomscroll_sessions_per_night', 'avg_doomscroll_session_minutes',
       'sleep_hours_per_night', 'sleep_latency_minutes',
       'number_of_night_wakeups', 'caffeine_intake_mg_per_day',
       'anxiety_score', 'stress_score', 'sleep_quality_score',
       'daytime_fatigue_score', 'number_of_news_apps_used',
       'phone_checks_per_night', 'exercise_minutes_per_day',
       'days_since_last_digital_detox', 'weekly_sleep_debt_hours', 'gender',
       'occupation_status', 'country_region', 'primary_device_used_at_night',
       'bedtime_routine_type', 'uses_night_mode', 'keeps_phone_in_bedroom',
       'consumes_negative_news_content', 'uses_sleep_tracking_app',
       'doomscroller', 'sleep_quality_category'],
      dtype='object')

In [154]:
#Need to make sure that the columns that i am using are predictors

# Can only use .corr() on numerical variables
Sleep_numerical = Sleep_clean.select_dtypes(include = ['int64','float64'])

# checking the correlation of the numerical datatypes
Sleep_numerical.corr()

target_corr = Sleep_numerical.corr()['sleep_hours_per_night'].drop('sleep_hours_per_night').abs().sort_values(ascending=False)
target_corr

weekly_sleep_debt_hours           0.894640
bedtime_screen_time_minutes       0.440230
doomscroll_sessions_per_night     0.360574
sleep_quality_score               0.345901
sleep_latency_minutes             0.343024
total_daily_screen_time_hours     0.250448
daytime_fatigue_score             0.180386
number_of_night_wakeups           0.177735
anxiety_score                     0.176443
doomscroller                      0.171761
phone_checks_per_night            0.121422
exercise_minutes_per_day          0.116769
avg_doomscroll_session_minutes    0.102479
stress_score                      0.078255
uses_night_mode                   0.063180
consumes_negative_news_content    0.042159
age                               0.039502
days_since_last_digital_detox     0.032871
caffeine_intake_mg_per_day        0.029061
number_of_news_apps_used          0.019988
keeps_phone_in_bedroom            0.004576
uses_sleep_tracking_app           0.003795
Name: sleep_hours_per_night, dtype: float64

I am seeing that weekly_sleep_debt_hours is pretty correlated with sleep_hours_per_night. I am going to choose to drop that as a predictor for my upcoming model.

Additionally, while perusing the data I see that sleep_quality_score, sleep_quality_category, and daytime_fatigue_score all seem to be based on the sleep someone is having every and not a predictor of the sleep (i.e. bedtime_screen_time_minutes). For this model we are predicting sleep_quality_score so I am dropping the remaining of these categories.

To recap I will be dropping: weekly_sleep_debt_hours, daytime_fatigue_score, and sleep_quality_category.

See code block below:

In [155]:
#Maddie copy this line for your regression
Sleep_clean = Sleep_clean.drop(columns = ['weekly_sleep_debt_hours', 'daytime_fatigue_score', 'sleep_quality_category'])

In [156]:
Sleep_clean.columns

Index(['age', 'bedtime_screen_time_minutes', 'total_daily_screen_time_hours',
       'doomscroll_sessions_per_night', 'avg_doomscroll_session_minutes',
       'sleep_hours_per_night', 'sleep_latency_minutes',
       'number_of_night_wakeups', 'caffeine_intake_mg_per_day',
       'anxiety_score', 'stress_score', 'sleep_quality_score',
       'number_of_news_apps_used', 'phone_checks_per_night',
       'exercise_minutes_per_day', 'days_since_last_digital_detox', 'gender',
       'occupation_status', 'country_region', 'primary_device_used_at_night',
       'bedtime_routine_type', 'uses_night_mode', 'keeps_phone_in_bedroom',
       'consumes_negative_news_content', 'uses_sleep_tracking_app',
       'doomscroller'],
      dtype='object')

Okay, sweet, I feel good with these predictors. I am going to seperate sleep_quality_score as my outcome and everything else as my predictor, do my split, and then sort my columns into types.

In [157]:
#giving each of the scores a class label
label_names = {2: "Score 2", 3: "Score 3", 4: "Score 4", 5: "Score 5"}
classes = np.array(sorted(label_names))

In [158]:
#Outcome vs predictors seperation 
y = Sleep_clean['sleep_quality_score']
X = Sleep_clean.drop(columns = 'sleep_quality_score')

In [159]:
#train test split (20/80 as hannah ordered)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size =0.2, stratify=y, random_state = 12
)

In [ ]:
#adding a validation set within the training set
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train,
    test_size=0.20,        
    stratify=y_train,      
    random_state=12
)

In [161]:
#Check what had already been dumified
already_dumified = []

for col in X_train.columns:
    vals = set(X_train[col].dropna().unique())
    if vals <= {0, 1} and len(vals) == 2:
        already_dumified.append(col)

print(already_dumified)

['uses_night_mode', 'keeps_phone_in_bedroom', 'consumes_negative_news_content', 'uses_sleep_tracking_app', 'doomscroller']


In [162]:
#Categorical predictors
Cat_predictors = X_train.select_dtypes(include=['object', 'string']).columns

In [163]:
Num_predictors = []
for col in X_train:
    if X_train[col].dtype in ['int64', 'float64'] and col not in already_dumified:
        Num_predictors.append(col)


In [164]:
# Preprocessing - Column Transformation
num_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, Num_predictors),
    ('bin', 'passthrough', already_dumified),
    ('cat', OneHotEncoder(drop='first', handle_unknown='warn'), Cat_predictors)
])

In [165]:
pipe_ovr = Pipeline(steps=[
('transformers', preprocessor),
("model", OneVsRestClassifier(LogisticRegression(max_iter=5000, random_state=123)))
])

pipe_multi = Pipeline(steps=[
('transformers', preprocessor),
("model", LogisticRegression(max_iter=5000, random_state=123))
])

pipe_ovr.fit(X_fit, y_fit)
pipe_multi.fit(X_fit, y_fit)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('transformers', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[float64](4,)","[2.,3.,4.,5.]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](25,)","['age','bedtime_screen_time_minutes','total_daily_screen_time_hours',..., 'consumes_negative_news_content','uses_sleep_tracking_app','doomscroller']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,25
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('bin', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specified

In [166]:
print(len(pipe_ovr['transformers'].get_feature_names_out()))
print(len(pipe_multi['transformers'].get_feature_names_out()))

44
44


This is great - we wanted 44 different predictors

In [167]:
# Checking out what our predictions were for pipe_ovr
y_train_pred = pipe_ovr.predict(X_train)
y_test_pred = pipe_ovr.predict(X_test)
print(y_test_pred)

[5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 4. 5. 5. 5. 5. 5. 4.
 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 4. 5. 5. 5. 5. 5. 5. 5. 5.
 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 4. 5. 5. 5.
 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 4. 5. 4. 5. 5. 5.
 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 4. 5. 5.
 5. 4. 5. 5. 5. 5. 3. 5. 5. 5. 5. 5. 5. 5. 4. 5. 5. 5. 5. 5. 5. 5. 5. 5.
 5. 5. 5. 5. 5. 5. 4. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5.
 4. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 4. 5. 5. 5. 5. 5.]


In [168]:
# Checking out what our predictions were for pipe_multi
y_train_pred = pipe_multi.predict(X_train)
y_test_pred = pipe_multi.predict(X_test)

print(y_test_pred)

[5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 4. 5. 5. 5. 5. 5. 4.
 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 4. 5. 5. 5. 5. 5. 5. 5. 5.
 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 4. 5. 5. 5.
 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 4. 5. 4. 5. 5. 5.
 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 4. 5. 5.
 5. 3. 5. 5. 5. 5. 3. 5. 5. 5. 5. 5. 5. 5. 4. 5. 5. 5. 5. 5. 5. 5. 5. 5.
 5. 5. 5. 5. 5. 5. 4. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5.
 4. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 3. 5. 5. 5. 5. 4. 5. 5. 5. 5. 5.]


In [169]:
baseline = DummyClassifier(strategy="prior")
baseline.fit(X_fit, y_fit)
p_dummy = baseline.predict_proba(X_val)
y_dummy = baseline.predict(X_val)
baseline_results = pd.DataFrame([{
    "Model": "Class-frequency baseline",
    "Validation log loss": log_loss(y_val, p_dummy, labels=classes),
    "Validation accuracy": accuracy_score(y_val, y_dummy),
    "Validation macro F1": f1_score(
    y_val, y_dummy, average="macro", zero_division=0
    )
}])
models = {"OvR": pipe_ovr, "Multinomial": pipe_multi}
model_rows = []
for name, model in models.items():
    p_fit = model.predict_proba(X_fit)
    p_val = model.predict_proba(X_val)
    pred_val = model.predict(X_val)
    model_rows.append({
        "Model": name,
        "Training log loss": log_loss(y_fit, p_fit, labels=classes),
        "Validation log loss": log_loss(y_val, p_val, labels=classes),
        "Validation accuracy": accuracy_score(y_val, pred_val),
        "Validation macro F1": f1_score(
        y_val, pred_val, average="macro", zero_division=0
        )
    })
results = pd.DataFrame(model_rows)
display(baseline_results.round(4))
display(results.round(4))

,Model,Validation log loss,Validation accuracy,Validation macro F1
0,Class-frequency baseline,0.4896,0.8636,0.2317


,Model,Training log loss,Validation log loss,Validation accuracy,Validation macro F1
0,OvR,0.3047,0.4098,0.8636,0.3229
1,Multinomial,0.2919,0.4179,0.8506,0.3158


#### Testing the models

In [170]:
best_name = results.loc[results["Validation log loss"].idxmin(), "Model"]
print("Selected using validation log loss:", best_name)
display(results.loc[results["Model"] == best_name].round(4))
# Refit the selected configuration on all 21 original training people.
# clone removes fitted state; the scaler is also refitted here.
final_model = clone(models[best_name])
final_model.fit(X_train, y_train)
print("Final model fitted. Its configuration is now fixed.")

Selected using validation log loss: OvR


,Model,Training log loss,Validation log loss,Validation accuracy,Validation macro F1
0,OvR,0.3047,0.4098,0.8636,0.3229


Final model fitted. Its configuration is now fixed.


In [171]:
p_test = final_model.predict_proba(X_test)
y_pred_test = final_model.predict(X_test)
final_results = pd.DataFrame([{
"Selected model": best_name,
"Test accuracy": accuracy_score(y_test, y_pred_test),
"Test macro F1": f1_score(
y_test, y_pred_test, average="macro", zero_division=0
),
"Test log loss": log_loss(y_test, p_test, labels=classes)
}])
display(final_results.round(4))

,Selected model,Test accuracy,Test macro F1,Test log loss
0,OvR,0.8542,0.2804,0.4634


### Trying a less granular outcome valriable for sleep quality score

In [172]:
Sleep_clean['sleep_quality_score'].value_counts()

sleep_quality_score
5.0    830
4.0    100
3.0     25
2.0      5
Name: count, dtype: int64

Because our sleep quality data is so skewed towards high values of sleep quality, it is going to make it hard for our model to predict low sleep quality scores which are rare in our data. This is why we see such a high accuracy score but log loss and macro F1 remain low. Below I am going to re-run the model on a grouped version of this outcome variable to look at the following classifications: 'Poor' where sleep quality score is 2 or 3, 'Good' where sleep quality score is 4, and 'Best' where sleep quality score is 5.

In [173]:
#creating the three classes
replace_map = {2: 0,
               3: 0,
               4: 1,
               5: 2}

Sleep_clean["sleep_quality_cat"] = Sleep_clean["sleep_quality_score"].map(replace_map)
label_names = {0: "Poor", 1: "Good", 2: "Best"}
classes = np.array(sorted(label_names))

In [174]:
#Outcome vs predictors seperation 
y = Sleep_clean['sleep_quality_cat']
X = Sleep_clean.drop(columns = ['sleep_quality_score', 'sleep_quality_cat'])

#train test split (20/80 as hannah ordered)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size =0.2, stratify=y, random_state = 12
)

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train,
    test_size=0.20,        
    stratify=y_train,      
    random_state=12
)

In [175]:
Cat_predictors = X_train.select_dtypes(include=['object', 'string']).columns

Num_predictors = []
for col in X_train:
    if X_train[col].dtype in ['int64', 'float64'] and col not in already_dumified:
        Num_predictors.append(col)

In [176]:
num_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, Num_predictors),
    ('bin', 'passthrough', already_dumified),
    ('cat', OneHotEncoder(drop='first', handle_unknown='warn'), Cat_predictors)
])

In [177]:
pipe_ovr = Pipeline(steps=[
('transformers', preprocessor),
("model", OneVsRestClassifier(LogisticRegression(max_iter=5000, random_state=123)))
])

pipe_multi = Pipeline(steps=[
('transformers', preprocessor),
("model", LogisticRegression(max_iter=5000, random_state=123))
])

pipe_ovr.fit(X_fit, y_fit)
pipe_multi.fit(X_fit, y_fit)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('transformers', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[0,1,2]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](25,)","['age','bedtime_screen_time_minutes','total_daily_screen_time_hours',..., 'consumes_negative_news_content','uses_sleep_tracking_app','doomscroller']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,25
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('bin', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns 

In [178]:
baseline = DummyClassifier(strategy="prior")
baseline.fit(X_fit, y_fit)
p_dummy = baseline.predict_proba(X_val)
y_dummy = baseline.predict(X_val)
baseline_results = pd.DataFrame([{
    "Model": "Class-frequency baseline",
    "Validation log loss": log_loss(y_val, p_dummy, labels=classes),
    "Validation accuracy": accuracy_score(y_val, y_dummy),
    "Validation macro F1": f1_score(
    y_val, y_dummy, average="macro", zero_division=0
    )
}])
models = {"OvR": pipe_ovr, "Multinomial": pipe_multi}
model_rows = []
for name, model in models.items():
    p_fit = model.predict_proba(X_fit)
    p_val = model.predict_proba(X_val)
    pred_val = model.predict(X_val)
    model_rows.append({
        "Model": name,
        "Training log loss": log_loss(y_fit, p_fit, labels=classes),
        "Validation log loss": log_loss(y_val, p_val, labels=classes),
        "Validation accuracy": accuracy_score(y_val, pred_val),
        "Validation macro F1": f1_score(
        y_val, pred_val, average="macro", zero_division=0
        )
    })
results = pd.DataFrame(model_rows)
display(baseline_results.round(4))
display(results.round(4))

,Model,Validation log loss,Validation accuracy,Validation macro F1
0,Class-frequency baseline,0.4732,0.8636,0.3089


,Model,Training log loss,Validation log loss,Validation accuracy,Validation macro F1
0,OvR,0.3096,0.4077,0.8701,0.4154
1,Multinomial,0.3036,0.4091,0.8571,0.4091


In [179]:
best_name = results.loc[results["Validation log loss"].idxmin(), "Model"]
print("Selected using validation log loss:", best_name)
display(results.loc[results["Model"] == best_name].round(4))
# Refit the selected configuration on all 21 original training people.
# clone removes fitted state; the scaler is also refitted here.
final_model = clone(models[best_name])
final_model.fit(X_train, y_train)
print("Final model fitted. Its configuration is now fixed.")

Selected using validation log loss: OvR


,Model,Training log loss,Validation log loss,Validation accuracy,Validation macro F1
0,OvR,0.3096,0.4077,0.8701,0.4154


Final model fitted. Its configuration is now fixed.


In [180]:
p_test = final_model.predict_proba(X_test)
y_pred_test = final_model.predict(X_test)
final_results = pd.DataFrame([{
"Selected model": best_name,
"Test accuracy": accuracy_score(y_test, y_pred_test),
"Test macro F1": f1_score(
y_test, y_pred_test, average="macro", zero_division=0
),
"Test log loss": log_loss(y_test, p_test, labels=classes)
}])
display(final_results.round(4))

,Selected model,Test accuracy,Test macro F1,Test log loss
0,OvR,0.8438,0.3504,0.4039
